In [0]:
#Load Config and Setup Enviorment Variables
# state_under_test = "paymentPending"
# state_under_test = "appealSubmitted"
# state_under_test = "awaitingRespondentEvidence(a)"
# state_under_test = "awaitingRespondentEvidence(b)"
# state_under_test = "caseUnderReview"
# state_under_test = "reasonsForAppealSubmitted"
state_under_test = "listing"


from pyspark.sql import functions as F
from pyspark.sql.functions import *
from delta.tables import DeltaTable

config = spark.read.option("multiline", "true").json("dbfs:/configs/config.json")
env_name = config.first()["env"].strip().lower()
lz_key = config.first()["lz_key"].strip().lower()
 
# print(f"env_code: {lz_key}")  # This won't be redacted
# print(f"env_name: {env_name}")  # This won't be redacted
 
KeyVault_name = f"ingest{lz_key}-meta002-{env_name}"
# print(f"KeyVault_name: {KeyVault_name}")
 
# Service principal credentials
client_id = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-CLIENT-ID")
client_secret = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-CLIENT-SECRET")
tenant_id = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-TENANT-ID")
 
# Storage account names
curated_storage = f"ingest{lz_key}curated{env_name}"
checkpoint_storage = f"ingest{lz_key}xcutting{env_name}"
raw_storage = f"ingest{lz_key}raw{env_name}"
landing_storage = f"ingest{lz_key}landing{env_name}"
external_storage = f"ingest{lz_key}external{env_name}"
  
# Spark config for curated storage (Delta table)
spark.conf.set(f"fs.azure.account.auth.type.{curated_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{curated_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{curated_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{curated_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{curated_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
 
# Spark config for checkpoint storage
spark.conf.set(f"fs.azure.account.auth.type.{checkpoint_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{checkpoint_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{checkpoint_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{checkpoint_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{checkpoint_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
 
# Spark config for checkpoint storage
spark.conf.set(f"fs.azure.account.auth.type.{raw_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{raw_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{raw_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{raw_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{raw_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
 
# Spark config for checkpoint storage
spark.conf.set(f"fs.azure.account.auth.type.{landing_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{landing_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{landing_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{landing_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{landing_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
 
 
# Spark config for checkpoint storage
spark.conf.set(f"fs.azure.account.auth.type.{external_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{external_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{external_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{external_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{external_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
  
# Setting variables for use in subsequent cells
bronze_path = f"abfss://bronze@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/"
silver_path = f"abfss://silver@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/"
audit_path = f"abfss://silver@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/AUDIT/{state_under_test}"
gold_path = f"abfss://gold@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/{state_under_test}"
 
 
# Print all variables
# variables = {
#     # "read_hive": read_hive,
    
#     "bronze_path": bronze_path,
#     "silver_path": silver_path,
#     "audit_path": audit_path,
#     "gold_path": gold_path,
#     "key_vault": KeyVault_name,
#     "AppealState": state_under_test
 
# }
 
# display(variables)

import json

#Get Latest Json Folder
json_location = dbutils.fs.ls(f"{gold_path}/")[-1]
latest_json_location = json_location.name
dbutils.fs.ls(f"{gold_path}/{latest_json_location}")

#Set Paths
try: 
    json_path = f"{gold_path}/{latest_json_location}/JSON/"
    json_failed_path = f"{gold_path}/{latest_json_location}/INVALID_JSON/"
    M1_silver = f"{silver_path}/silver_appealcase_detail"
    M1_bronze = f"{bronze_path}/bronze_appealcase_crep_rep_floc_cspon_cfs"
    M2_bronze = f"{bronze_path}/bronze_appealcase_caseappellant_appellant"
    M2_silver = f"{silver_path}/silver_caseapplicant_detail"
    M3_silver = f"{silver_path}/silver_status_detail"
    C = f"{silver_path}/silver_appealcategory_detail"
    bhc = f"{bronze_path}/bronze_hearing_centres"
    bat = f"{bronze_path}/bronze_appealtype" 
    docsr = f"{bronze_path}/bronze_documentsreceived"   
    # apl_audit = f"{audit_path}/apl_active_payment_pending_cr_audit_table/"
    sh =  f"{silver_path}/silver_history_detail"
except:
    print(f"Error during fetch: {str(e)}")

#Create and Load Dataframes
json_data = spark.read.format("json").load(json_path)
json_failed_data = spark.read.format("json").load(json_failed_path)
M1_silver = spark.read.format("delta").load(M1_silver)
M1_bronze = spark.read.format("delta").load(M1_bronze)
M2_bronze = spark.read.format("delta").load(M2_bronze)
M2_silver = spark.read.format("delta").load(M2_silver)
M3_silver = spark.read.format("delta").load(M3_silver)
C = spark.read.format("delta").load(C)
bhc = spark.read.format("delta").load(bhc)
bat = spark.read.format("delta").load(bat)
docsr = spark.read.format("delta").load(docsr)
# apl_audit = spark.read.format("delta").load(apl_audit)
sh_audit = spark.read.format("delta").load(sh)

#Can be removed later, added to allow developing of code in this notebook to begin with before moving to func files
from pyspark.sql.functions import (
    col, when, lit, array, struct, collect_list, 
    max as spark_max, date_format, row_number, expr, 
    size, udf, coalesce, concat_ws, concat, trim, year, split, datediff,
    collect_set, current_timestamp,transform, first, array_contains
)

# --- patch diagnostics: collected from every patch cell, reported at the end ---
PATCH_LOG = []


# Data Patching - For Data Cut : 07-04-2025

In [0]:
###############################
#UPDATE BRONZE DATA SCRIPT FOR PAYMENT PENDING.
#
#NOTE: The below code will update bronze data that will not pass the DQ expecation checks due to
#issues in the data that will be resolved before live but are needeed to get all the data through the checks
#and sent to CCD in the mean time
###############################
from pyspark.sql.functions import *
from delta.tables import DeltaTable

######################
#TO FIX PP DATA FROM FIRST STAGING DATA CUT (now superseeded due to new data cut)
######################
# BirthDate / appellantDateOfBirth

# bronze_table = DeltaTable.forName(spark,"ariadm_active_appeals.bronze_appealcase_caseappellant_appellant")

# display(bronze_table.toDF().filter(col("CaseNo").isin("HU/00278/2025", "HU/00455/2025", "HU/00472/2025" )).select("CaseNo", "BirthDate"))

# bronze_table.update(
#     condition=col("CaseNo").isin("HU/00278/2025", "HU/00455/2025", "HU/00472/2025"),
#     set={"BirthDate": lit("2000-02-01T00:00:00Z")}
# )

# display(bronze_table.toDF().filter(col("CaseNo").isin("HU/00278/2025", "HU/00455/2025", "HU/00472/2025" )).select("CaseNo", "BirthDate"))


# #################
# #valid_appellantNationalitiesDescription_not_null
# #and
# #valid_appellantNationalities_not_null
# #NationalityId
# #Where No mapping required for 201/203

# bronze_table = DeltaTable.forName(spark, "ariadm_active_appeals.bronze_appealcase_crep_rep_floc_cspon_cfs")

# display(bronze_table.toDF().filter(col("CaseNo").isin("HU/00302/2025", "HU/00569/2025", "HU/00586/2025","HU/00560/2025" )).select("CaseNo", "NationalityId"))

# bronze_table.update(
#     condition=col("CaseNo").isin("HU/00302/2025", "HU/00569/2025", "HU/00586/2025","HU/00560/2025"),
#     set={"NationalityId": lit("41")}
# )

# display(bronze_table.toDF().filter(col("CaseNo").isin("HU/00302/2025", "HU/00569/2025", "HU/00586/2025","HU/00560/2025" )).select("CaseNo", "NationalityId"))

# #################
# #valid_oocAddressLine1 valid_oocAddressLine2
# #changing null values to actual values

# bronze_table = DeltaTable.forName(spark, "ariadm_active_appeals.bronze_appealcase_crep_rep_floc_cspon_cfs")
# display(bronze_table.toDF().filter(col("CaseNo").isin("HU/00185/2025", "HU/02151/2024")).select("CaseNo", "CaseRep_Address1", "CaseRep_Address2", "CaseRep_Address3", "CaseRep_Address4" , "CaseRep_Address5", "CaseRep_Postcode"))

# bronze_table.update(
#     condition=col("CaseNo").isin("HU/00185/2025", "HU/02151/2024"),
#     set={"CaseRep_Address1": lit("925 Lisa Plains Apt. 642X"),
#          "CaseRep_Address2" : lit("Hill SquareX"),
#         "CaseRep_Address3" : lit("LynchhavenX"),
#         "CaseRep_Address4" : lit("AustraliaX"),
#         "CaseRep_Address5" : lit("NLX"),
#         # "CaseRep_Postcode" : lit("Hill SquareX"),
         
#          }
# )


# display(bronze_table.toDF().filter(col("CaseNo").isin("HU/00185/2025", "HU/02151/2024")).select("CaseNo", "CaseRep_Address1", "CaseRep_Address2", "CaseRep_Address3", "CaseRep_Address4" , "CaseRep_Address5", "CaseRep_Postcode"))


# #################
# #valid_oocAddressLine1 valid_oocAddressLine2
# #change in ooc4 ReunionX to GuamX

# bronze_table = DeltaTable.forName(spark, "ariadm_active_appeals.bronze_appealcase_crep_rep_floc_cspon_cfs")
# display(bronze_table.toDF().filter(col("CaseNo").isin("HU/02191/2024", "HU/01475/2024")).select("CaseNo", "CaseRep_Address1", "CaseRep_Address2", "CaseRep_Address3", "CaseRep_Address4" , "CaseRep_Address5", "CaseRep_Postcode"))

# bronze_table.update(
#     condition=col("CaseNo").isin("HU/01475/2024", "HU/02191/2024"),
#     set={
#         "CaseRep_Address4" : lit("AustraliaX")                 
#          }
# )


# display(bronze_table.toDF().filter(col("CaseNo").isin("HU/02191/2024", "HU/01475/2024")).select("CaseNo", "CaseRep_Address1", "CaseRep_Address2", "CaseRep_Address3", "CaseRep_Address4" , "CaseRep_Address5", "CaseRep_Postcode"))



# ################################

# cases_to_update = ['EA/02806/2023',
# 'HU/00575/2025',
# 'HU/00581/2025',
# 'HU/00447/2025',
# 'HU/00574/2023',
# 'HU/00591/2025',
# 'EA/00588/2025',
# 'EA/00551/2025',
# 'HU/00304/2025',
# 'EA/00495/2025',
# 'EA/00560/2025',
# 'EA/06826/2022',
# 'EA/00490/2025',
# 'EA/00554/2025',
# 'EA/01778/2024',
# 'HU/00562/2025',
# 'EA/00552/2025',
# 'HU/00511/2025',
# 'EA/00483/2025',
# 'EA/09676/2022',
# 'EA/00493/2025',
# 'HU/00224/2025',
# 'EA/00496/2025',
# 'EA/00538/2025',
# 'EA/00558/2025',
# 'EA/08372/2022',
# 'HU/00822/2024',
# 'HU/00574/2025',
# 'EA/02065/2024',
# 'HU/00442/2025',
# 'EA/00586/2025',
# 'HU/00590/2025',
# 'HU/00569/2025',
# 'EA/00557/2025',
# 'HU/02346/2024',
# 'EA/00562/2025',
# 'HU/00573/2025',
# 'HU/00571/2025',
# 'EA/01893/2023',
# 'EA/00584/2025',
# 'HU/00579/2025',
# 'HU/00555/2025',
# 'HU/00583/2025',
# 'EA/00591/2025',
# 'EA/00556/2025',
# 'EA/00497/2025',
# 'HU/01972/2023',
# 'EA/00437/2025',
# 'HU/00577/2025',
# 'EA/00585/2025',
# 'HU/00252/2025',
# 'HU/00557/2025',
# 'EA/00485/2025',
# 'HU/00563/2025',
# 'HU/00278/2025',
# 'EA/00559/2025',
# 'EA/00553/2025',
# 'HU/00578/2025',
# 'HU/00445/2025',
# 'HU/00325/2025',
# 'EA/00555/2025',
# 'HU/00572/2025',
# 'HU/00582/2025',
# 'EA/00587/2025',
# 'HU/00638/2024',
# 'HU/00453/2025',
# "EA/00289/2025"
# ]


# bronze_table = DeltaTable.forName(spark, "ariadm_active_appeals.bronze_appealcase_caseappellant_appellant")

# display(bronze_table.toDF().filter(col("CaseNo").isin(cases_to_update)).select("CaseNo", "Appellant_Address4"))

# bronze_table.update(
#     condition=col("CaseNo").isin(cases_to_update),
#     set={
#         "Appellant_Address4" : lit("AustraliaX")                 
#          }
# )


# display(bronze_table.toDF().filter(col("CaseNo").isin(cases_to_update)).select("CaseNo", "Appellant_Address4"))




# Data Patching - For Data cut from : 02-02-2026

In [0]:
#TABLES
# bronze_table_to_update = "ariadm_active_appeals.bronze_appealcase_crep_rep_floc_cspon_cfs"
# bronze_table_to_update = "ariadm_active_appeals.bronze_appealcase_caseappellant_appellant"

state_to_update_1 = "paymentpending_gold.stg_main_paymentPending_validation"
state_to_update_2 = "appealsubmitted_gold.stg_main_appeal_submitted_validation"
state_to_update_3 = "awaitingrespondentevidencea_gold.stg_main_awaiting_respondent_evidence_a_validation"
state_to_update_4 = "awaitingrespondentevidenceb_gold.stg_main_awaiting_respondent_evidence_b_validation"

state_to_update_5 = "caseUnderReview_gold.stg_main_case_under_review_validation"
state_to_update_6 = "reasonsForAppealSubmitted_gold.stg_main_reasons_for_appeal_submitted_validation"
state_to_update_7 = "listing_gold.stg_main_listing_validation"

state_to_update_8 = "prepareforhearing_gold.stg_main_prepare_for_hearing_validation"
state_to_update_8 = "decision_gold.stg_main_decision_validation"
state_to_update_10 = "decided_a_gold.stg_main_decided_a_validation"



In [0]:
##############################
#valid_countryGovUkOocAdminJ - patch : "AppellantCountryId"  (DIRECT bronze patch, no gold read)
# 2026-07-23 (Peter's decision): the 7 unmapped ids -- 106 Marshall Islander / 110 Micronesian /
# 196 British Overseas Citizen / 201 Serbia&Montenegro / 203 (NULL) / 207 British National(Overseas) /
# 211 Stateless -- CORRECTLY have no countryGovUkOocAdminJ mapping (citizenship statuses that do not exist
# in CCD or ARIA PROD). The DQ rule is right; we patch OUR data cut so these OUT-OF-COUNTRY cases pass and
# reach CCD (data corrected before live). Lever = AppellantCountryId: getCountryApp maps it via
# bronze_countries_postal (the address/searchCountry path only runs when the id == 0, so address patches
# were inert). Set it to 57 = 'French' -> ISO 'FR', a valid OOC country in the rule's allow-list.
# Gated on the 23 KNOWN-FAILING OOC CaseNos only -> no in-country case (which must keep
# countryGovUkOocAdminJ NULL) is ever touched.
# NB: id 151 (Solomon Islands -> SB) is fixed separately via the mapping doc (APPENDIX-RefData), NOT here.
##############################

bronze_table_to_update = "ariadm_active_appeals.bronze_appealcase_caseappellant_appellant"
cg_bt = DeltaTable.forName(spark, bronze_table_to_update)
cg_bs = spark.table(bronze_table_to_update)

# 23 OOC cases failing valid_countryGovUkOocAdminJ (AppellantCountryId -> 'NO MAPPING REQUIRED')
countrygov_cases = [
    "HU/00652/2025","EA/00396/2024","EA/00873/2024","EA/01209/2024","EA/01222/2024",
    "EA/01557/2024","EA/02059/2024","EA/04446/2019","EA/00040/2025","EA/00627/2025",
    "EA/00709/2025","EA/00942/2024","HU/01957/2024","HU/00638/2025","EA/00776/2025",
    "EA/09676/2022","EA/00086/2025","EA/00647/2025","EA/01940/2024","EA/02116/2024",
    "EA/03368/2023","HU/00415/2025","HU/00671/2024",
]

patch_cond = col("CaseNo").isin(countrygov_cases)

#Display Before
display(cg_bs.filter(patch_cond).select("CaseNo", "AppellantCountryId"))

_n = cg_bs.filter(patch_cond).select("CaseNo").distinct().count()

cg_bt.update(
    condition=patch_cond,
    set={
        "AppellantCountryId" : lit(57)
         }
)

#Display After
display(cg_bs.filter(patch_cond).select("CaseNo", "AppellantCountryId"))

_pl = globals().setdefault("PATCH_LOG", [])
_pl.append({"rule": "valid_countryGovUkOocAdminJ", "table": bronze_table_to_update, "state": "ALL (direct bronze)", "cases_patched": _n})
print(f"   -> patched AppellantCountryId=57 (French/FR) for {_n} OOC case(s) failing countryGovUkOocAdminJ")


In [0]:
##############################
#valid_oocrCountryGovUkAdminJ - patch CaseRep_PostCode
#oocLrCountryGovUkAdminJ
##############################


#Setup
checks = {}
bronze_table_to_update = "ariadm_active_appeals.bronze_appealcase_crep_rep_floc_cspon_cfs"

#Add the Gold Tables to the list below for any States that need updating
state_to_update_2 = "appealsubmitted_gold.stg_main_appeal_submitted_validation"
state_to_update_3 = "awaitingrespondentevidencea_gold.stg_main_awaiting_respondent_evidence_a_validation"
state_to_update_4 = "awaitingrespondentevidenceb_gold.stg_main_awaiting_respondent_evidence_b_validation"
state_to_update_5 = "caseunderreview_gold.stg_main_case_under_review_validation"
state_to_update_6 = "reasonsForAppealSubmitted_gold.stg_main_reasons_for_appeal_submitted_validation"
state_to_update_7 = "listing_gold.stg_main_listing_validation"
state_to_update_8 = "prepareforhearing_gold.stg_main_prepare_for_hearing_validation"
state_to_update_9 = "remitted_gold.stg_main_remitted_validation"
state_to_update_10 = "decideda_gold.stg_main_decided_a_validation"
state_to_update_11 = "decision_gold.stg_main_decision_validation"
state_to_update_12 = "ended_gold.stg_main_ended_validation"
state_to_update_13 = "ftpadecided_gold.stg_main_ftpadecided_validation"
state_to_update_14 = "ftpasubmitteda_gold.stg_main_ftpa_submitted_a_validation"
state_to_update_15 = "ftpasubmittedb_gold.stg_main_ftpa_submitted_b_validation"
# states_to_update = [state_to_update_5, state_to_update_6,state_to_update_7]
# states_to_update = [state_to_update_2, state_to_update_3,state_to_update_4]
states_to_update = [state_to_update_2,state_to_update_3,state_to_update_4, state_to_update_5,state_to_update_6,state_to_update_7,state_to_update_8,state_to_update_9, state_to_update_10, state_to_update_11, state_to_update_12, state_to_update_13, state_to_update_14, state_to_update_15]

checks["valid_oocrCountryGovUkAdminJ"] = (
    """(
        (dv_representation = 'LR' AND legalRepHasAddress <=> 'No' AND oocLrCountryGovUkAdminJ IS NOT NULL)
        OR
        (dv_representation = 'LR' AND legalRepHasAddress <=> 'Yes' AND oocLrCountryGovUkAdminJ IS NULL)
        OR
        (dv_representation != 'LR' AND CaseRep_Address5 IS NULL)
    )"""
)

#####################
#Get Cases Failing Validation
#####################
dq_rules = "({0})".format(" AND ".join(checks.values()))


for state_table in states_to_update:
    print(f"Processing State : {state_table}")
    df_validation_filtered = spark.sql(f"""
    SELECT *
    FROM {state_table}
    WHERE CaseNo NOT IN (
    SELECT
        CaseNo
    FROM {state_table}
    WHERE {dq_rules})
    """)
    df_validation_filtered = df_validation_filtered.select("CaseNo", "oocLrCountryGovUkAdminJ")


    #####################
    #join with bronze data
    #####################
    #Join Failed Record in Gold With Bronze
    df_validation_filtered = df_validation_filtered.join(M2_bronze, on="CaseNo", how="inner")
    # display(df_validation_filtered.select("CaseNo",  "oocLrCountryGovUkAdminJ", "CaseRep_PostCode"))
    display(df_validation_filtered.select("CaseNo",  "oocLrCountryGovUkAdminJ"))

    #####################
    #Update Data
    #####################
    #Pull out case Numbers to patch
    case_nos_to_update = [row["CaseNo"] for row in df_validation_filtered.select(trim(col("CaseNo")).alias("CaseNo")).distinct().collect()]
    # --- patch diagnostic (auto) ---
    _pl = globals().setdefault("PATCH_LOG", [])
    _pl.append({"rule": ", ".join(checks.keys()), "table": bronze_table_to_update, "state": state_table, "cases_patched": len(case_nos_to_update)})
    print(f"   -> patched {len(case_nos_to_update)} case(s) for {', '.join(checks.keys())} in {state_table}")
    #set table to update
    bronze_table_to_update_df = DeltaTable.forName(spark, bronze_table_to_update)

    #Display Before    
    display(M1_bronze.filter(col("CaseNo").isin(case_nos_to_update)).select("CaseNo", "CaseRep_PostCode"))    

    #E3U 5EH
    #SW1A 1A
    bronze_table_to_update_df.update(
        condition=trim(col("CaseNo")).isin(case_nos_to_update),
        set={
            "CaseRep_PostCode" : lit("E3U 5EH")                 
             }
    )

    #Display After
    display(M1_bronze.filter(col("CaseNo").isin(case_nos_to_update)).select("CaseNo", "CaseRep_PostCode"))    

In [0]:
##############################
#valid_legalrepEmail_not_null - patch one of these : "Rep_Email" , "CaseRep_Email" , "FileSpecificEmail"
##############################


#Setup
checks = {}
bronze_table_to_update = "ariadm_active_appeals.bronze_appealcase_crep_rep_floc_cspon_cfs"
bronze_table = spark.table(bronze_table_to_update)

#Add the Gold Tables to the list below for any States that need updating
state_to_update_2 = "appealsubmitted_gold.stg_main_appeal_submitted_validation"
state_to_update_5 = "caseunderreview_gold.stg_main_case_under_review_validation"
state_to_update_6 = "reasonsForAppealSubmitted_gold.stg_main_reasons_for_appeal_submitted_validation"
state_to_update_7 = "listing_gold.stg_main_listing_validation"
state_to_update_8 = "prepareforhearing_gold.stg_main_prepare_for_hearing_validation"
state_to_update_9 = "remitted_gold.stg_main_remitted_validation"
state_to_update_10 = "decideda_gold.stg_main_decided_a_validation"
state_to_update_11 = "decision_gold.stg_main_decision_validation"
state_to_update_12 = "ended_gold.stg_main_ended_validation"
state_to_update_13 = "ftpadecided_gold.stg_main_ftpadecided_validation"
state_to_update_14 = "ftpasubmitteda_gold.stg_main_ftpa_submitted_a_validation"
state_to_update_15 = "ftpasubmittedb_gold.stg_main_ftpa_submitted_b_validation"

states_to_update = [state_to_update_2, state_to_update_5, state_to_update_6,state_to_update_7,state_to_update_8,state_to_update_9, state_to_update_10, state_to_update_11, state_to_update_12, state_to_update_13, state_to_update_14, state_to_update_15]
# states_to_update = [state_to_update_2]


checks["valid_legalrepEmail_not_null"] = "((dv_representation = 'LR' AND legalRepEmail IS NOT NULL AND legalRepEmail RLIKE r'^([a-zA-Z0-9_\\-\\.]+)@([a-zA-Z0-9_\\-\\.]+)\\.([a-zA-Z]{2,5})$') OR (dv_representation != 'LR' AND legalRepEmail IS NULL))"


#####################
#Get Cases Failing Validation
#####################
dq_rules = "({0})".format(" AND ".join(checks.values()))


for state_table in states_to_update:
    print(f"Processing State : {state_table}")
    df_validation_filtered = spark.sql(f"""
    SELECT *
    FROM {state_table}
    WHERE CaseNo NOT IN (
    SELECT
        CaseNo
    FROM {state_table}
    WHERE {dq_rules})
    """)
    df_validation_filtered = df_validation_filtered.select("CaseNo", "legalRepEmail")


    #####################
    #join with bronze data
    #####################
    #Join Failed Record in Gold With Bronze
    df_validation_filtered = df_validation_filtered.withColumn("CaseNo", trim(col("CaseNo"))).join(M1_bronze.withColumn("CaseNo", trim(col("CaseNo"))), on="CaseNo", how="inner")
    # display(df_validation_filtered)
    # display(df_validation_filtered.select("CaseNo",  "legalRepEmail", "CaseRep_Email"))

    #####################
    #Update Data
    #####################
    #Pull out case Numbers to patch
    case_nos_to_update = [row["CaseNo"] for row in df_validation_filtered.select(trim(col("CaseNo")).alias("CaseNo")).distinct().collect()]
    # --- patch diagnostic (auto) ---
    _pl = globals().setdefault("PATCH_LOG", [])
    _pl.append({"rule": ", ".join(checks.keys()), "table": bronze_table_to_update, "state": state_table, "cases_patched": len(case_nos_to_update)})
    print(f"   -> patched {len(case_nos_to_update)} case(s) for {', '.join(checks.keys())} in {state_table}")
    #set table to update
    bronze_table_to_update_df = DeltaTable.forName(spark, bronze_table_to_update)

    #Display Before
    # display(bronze_table)
    display(M1_bronze.filter(col("CaseNo").isin(case_nos_to_update)).select("CaseNo", "CaseRep_Email"))    

    bronze_table_to_update_df.update(
        condition=trim(col("CaseNo")).isin(case_nos_to_update),
        set={
           "CaseRep_Email" : lit("joexbloggs@fake.com")                 
            }
    )

    #Display After
    display(M1_bronze.filter(col("CaseNo").isin(case_nos_to_update)).select("CaseNo", "CaseRep_Email"))

In [0]:
##############################
#valid_oocAddressLine1 - patch : "CaseRep_Address1"
##############################


#Setup
checks = {}
bronze_table_to_update = "ariadm_active_appeals.bronze_appealcase_crep_rep_floc_cspon_cfs"

#Add the Gold Tables to the list below for any States that need updating
state_to_update_2 = "appealsubmitted_gold.stg_main_appeal_submitted_validation"
state_to_update_3 = "awaitingrespondentevidencea_gold.stg_main_awaiting_respondent_evidence_a_validation"
state_to_update_4 = "awaitingrespondentevidenceb_gold.stg_main_awaiting_respondent_evidence_b_validation"
state_to_update_5 = "caseunderreview_gold.stg_main_case_under_review_validation"
state_to_update_6 = "reasonsForAppealSubmitted_gold.stg_main_reasons_for_appeal_submitted_validation"
state_to_update_7 = "listing_gold.stg_main_listing_validation"
state_to_update_8 = "prepareforhearing_gold.stg_main_prepare_for_hearing_validation"
state_to_update_9 = "remitted_gold.stg_main_remitted_validation"
state_to_update_10 = "decideda_gold.stg_main_decided_a_validation"
state_to_update_11 = "decision_gold.stg_main_decision_validation"
state_to_update_12 = "ended_gold.stg_main_ended_validation"
state_to_update_13 = "ftpadecided_gold.stg_main_ftpadecided_validation"
state_to_update_14 = "ftpasubmitteda_gold.stg_main_ftpa_submitted_a_validation"
state_to_update_15 = "ftpasubmittedb_gold.stg_main_ftpa_submitted_b_validation"

states_to_update = [state_to_update_2, state_to_update_3,state_to_update_4, state_to_update_5, state_to_update_6,state_to_update_7,state_to_update_8, state_to_update_9, state_to_update_10, state_to_update_11, state_to_update_12, state_to_update_13, state_to_update_14, state_to_update_15]
# states_to_update = [state_to_update_2, state_to_update_3,state_to_update_4]

checks["valid_oocAddressLine1"] = (
        """(
            (dv_representation = 'LR' AND oocAddressLine1 IS NOT NULL AND legalRepHasAddress <=> 'No')
            OR
            (dv_representation = 'LR' AND oocAddressLine1 IS NULL AND legalRepHasAddress <=> 'Yes')
            OR
            (dv_representation != 'LR' AND oocAddressLine1 IS NULL)
        )"""
    )

#####################
#Get Cases Failing Validation
#####################
dq_rules = "({0})".format(" AND ".join(checks.values()))


for state_table in states_to_update:
    print(f"Processing State : {state_table}")
    df_validation_filtered = spark.sql(f"""
    SELECT *
    FROM {state_table}
    WHERE CaseNo NOT IN (
    SELECT
        CaseNo
    FROM {state_table}
    WHERE {dq_rules})
    """)
    df_validation_filtered = df_validation_filtered.select("CaseNo", "oocAddressLine1")


    #####################
    #join with bronze data
    #####################
    #Join Failed Record in Gold With Bronze
    df_validation_filtered = df_validation_filtered.withColumn("CaseNo", trim(col("CaseNo"))).join(M1_bronze.withColumn("CaseNo", trim(col("CaseNo"))), on="CaseNo", how="inner")
    display(df_validation_filtered.select("CaseNo",  "oocAddressLine1", "CaseRep_Address1"))

    #####################
    #Update Data
    #####################
    #Pull out case Numbers to patch
    case_nos_to_update = [row["CaseNo"] for row in df_validation_filtered.select(trim(col("CaseNo")).alias("CaseNo")).distinct().collect()]
    # --- patch diagnostic (auto) ---
    _pl = globals().setdefault("PATCH_LOG", [])
    _pl.append({"rule": ", ".join(checks.keys()), "table": bronze_table_to_update, "state": state_table, "cases_patched": len(case_nos_to_update)})
    print(f"   -> patched {len(case_nos_to_update)} case(s) for {', '.join(checks.keys())} in {state_table}")
    #set table to update
    bronze_table_to_update_df = DeltaTable.forName(spark, bronze_table_to_update)

    #Display Before    
    display(M1_bronze.filter(col("CaseNo").isin(case_nos_to_update)).select("CaseNo", "CaseRep_Address1"))    

    bronze_table_to_update_df.update(
        condition=trim(col("CaseNo")).isin(case_nos_to_update),
        set={
            "CaseRep_Address1" : lit("617 Joshua Park Apt. 191X")                 
             }
    )

    #Display After
    display(M1_bronze.filter(col("CaseNo").isin(case_nos_to_update)).select("CaseNo", "CaseRep_Address1"))    


In [0]:
##############################
#valid_oocAddressLine2 - patch : "CaseRep_Address2"
##############################


#Setup
checks = {}
bronze_table_to_update = "ariadm_active_appeals.bronze_appealcase_crep_rep_floc_cspon_cfs"

#Add the Gold Tables to the list below for any States that need updating
state_to_update_2 = "appealsubmitted_gold.stg_main_appeal_submitted_validation"
state_to_update_3 = "awaitingrespondentevidencea_gold.stg_main_awaiting_respondent_evidence_a_validation"
state_to_update_4 = "awaitingrespondentevidenceb_gold.stg_main_awaiting_respondent_evidence_b_validation"
state_to_update_5 = "caseunderreview_gold.stg_main_case_under_review_validation"
state_to_update_6 = "reasonsForAppealSubmitted_gold.stg_main_reasons_for_appeal_submitted_validation"
state_to_update_7 = "listing_gold.stg_main_listing_validation"
state_to_update_8 = "prepareforhearing_gold.stg_main_prepare_for_hearing_validation"
state_to_update_9 = "remitted_gold.stg_main_remitted_validation"
state_to_update_10 = "decideda_gold.stg_main_decided_a_validation"
state_to_update_11 = "decision_gold.stg_main_decision_validation"
state_to_update_12 = "ended_gold.stg_main_ended_validation"
state_to_update_13 = "ftpadecided_gold.stg_main_ftpadecided_validation"
state_to_update_14 = "ftpasubmitteda_gold.stg_main_ftpa_submitted_a_validation"
state_to_update_15 = "ftpasubmittedb_gold.stg_main_ftpa_submitted_b_validation"

states_to_update = [state_to_update_2, state_to_update_3,state_to_update_4, state_to_update_5, state_to_update_6,state_to_update_7,state_to_update_8,state_to_update_9, state_to_update_10, state_to_update_11, state_to_update_12, state_to_update_13, state_to_update_14, state_to_update_15]
# states_to_update = [state_to_update_2, state_to_update_3,state_to_update_4]

checks["valid_oocAddressLine2"] = (
        """(
            (dv_representation = 'LR' AND oocAddressLine2 IS NOT NULL AND legalRepHasAddress <=> 'No')
            OR
            (dv_representation = 'LR' AND oocAddressLine2 IS NULL AND legalRepHasAddress <=> 'Yes')
            OR
            (dv_representation != 'LR' AND oocAddressLine2 IS NULL)
        )"""
    )

#####################
#Get Cases Failing Validation
#####################
dq_rules = "({0})".format(" AND ".join(checks.values()))


for state_table in states_to_update:
    print(f"Processing State : {state_table}")
    df_validation_filtered = spark.sql(f"""
    SELECT *
    FROM {state_table}
    WHERE CaseNo NOT IN (
    SELECT
        CaseNo
    FROM {state_table}
    WHERE {dq_rules})
    """)
    df_validation_filtered = df_validation_filtered.select("CaseNo", "oocAddressLine2")


    #####################
    #join with bronze data
    #####################
    #Join Failed Record in Gold With Bronze
    df_validation_filtered = df_validation_filtered.withColumn("CaseNo", trim(col("CaseNo"))).join(M1_bronze.withColumn("CaseNo", trim(col("CaseNo"))), on="CaseNo", how="inner")
    display(df_validation_filtered.select("CaseNo",  "oocAddressLine2", "CaseRep_Address2"))

    #####################
    #Update Data
    #####################
    #Pull out case Numbers to patch
    case_nos_to_update = [row["CaseNo"] for row in df_validation_filtered.select(trim(col("CaseNo")).alias("CaseNo")).distinct().collect()]
    # --- patch diagnostic (auto) ---
    _pl = globals().setdefault("PATCH_LOG", [])
    _pl.append({"rule": ", ".join(checks.keys()), "table": bronze_table_to_update, "state": state_table, "cases_patched": len(case_nos_to_update)})
    print(f"   -> patched {len(case_nos_to_update)} case(s) for {', '.join(checks.keys())} in {state_table}")
    #set table to update
    bronze_table_to_update_df = DeltaTable.forName(spark, bronze_table_to_update)

    #Display Before    
    display(M1_bronze.filter(col("CaseNo").isin(case_nos_to_update)).select("CaseNo", "CaseRep_Address2"))    

    bronze_table_to_update_df.update(
        condition=trim(col("CaseNo")).isin(case_nos_to_update),
        set={
            "CaseRep_Address2" : lit("Thomas ValleyX")                 
             }
    )

    #Display After
    display(M1_bronze.filter(col("CaseNo").isin(case_nos_to_update)).select("CaseNo", "CaseRep_Address2"))    


In [0]:
##############################
#valid_appellantNationalities_not_null / valid_appellantNationalitiesDescription_not_null - patch : "NationalityId"
# NOTE: rule text below is copied from the DEPLOYED (main) dq_rules - that is what builds gold
# and what dq_validation flags. (The Active-Testing branch has a different lookup-based rule.)
# TODO: load the live rule from /Workspace/live/.../dq_rules instead of hardcoding.
##############################


#Setup
checks = {}
bronze_table_to_update = "ariadm_active_appeals.bronze_appealcase_crep_rep_floc_cspon_cfs"

#Add the Gold Tables to the list below for any States that need updating
state_to_update_2 = "appealsubmitted_gold.stg_main_appeal_submitted_validation"
state_to_update_3 = "awaitingrespondentevidencea_gold.stg_main_awaiting_respondent_evidence_a_validation"
state_to_update_4 = "awaitingrespondentevidenceb_gold.stg_main_awaiting_respondent_evidence_b_validation"
state_to_update_5 = "caseunderreview_gold.stg_main_case_under_review_validation"
state_to_update_6 = "reasonsForAppealSubmitted_gold.stg_main_reasons_for_appeal_submitted_validation"
state_to_update_7 = "listing_gold.stg_main_listing_validation"
state_to_update_8 = "prepareforhearing_gold.stg_main_prepare_for_hearing_validation"
state_to_update_9 = "remitted_gold.stg_main_remitted_validation"
state_to_update_10 = "decideda_gold.stg_main_decided_a_validation"
state_to_update_11 = "decision_gold.stg_main_decision_validation"
state_to_update_12 = "ended_gold.stg_main_ended_validation"
state_to_update_13 = "ftpadecided_gold.stg_main_ftpadecided_validation"
state_to_update_14 = "ftpasubmitteda_gold.stg_main_ftpa_submitted_a_validation"
state_to_update_15 = "ftpasubmittedb_gold.stg_main_ftpa_submitted_b_validation"

states_to_update = [state_to_update_2, state_to_update_3, state_to_update_4, state_to_update_5, state_to_update_6, state_to_update_7, state_to_update_8, state_to_update_9, state_to_update_10, state_to_update_11, state_to_update_12, state_to_update_13, state_to_update_14, state_to_update_15]

checks["valid_appellantNationalities_not_null"] = (
            """(
                (appellantNationalities IS NOT NULL)
                AND
                EXISTS(appellantNationalities, x -> x.value.code IN ('AF', 'AX', 'AL', 'DZ', 'AS', 'AD', 'AO', 'AI', 'AQ', 'AG', 'AR', 'AM', 'AW', 'AU', 'AT', 'AZ', 'BS', 'BH', 'BD', 'BB', 'BY', 'BE', 'BZ', 'BJ', 'BM', 'BT', 'BO', 'BQ', 'BA', 'BW', 'BV', 'BR', 'BC', 'VG', 'IO', 'BN', 'BG', 'BF', 'BI', 'KH', 'CM', 'CA', 'CV', 'KY', 'CF', 'TD', 'CL', 'CN', 'HK', 'MO', 'CX', 'CC', 'CO', 'KM', 'CG', 'CD', 'CK', 'CR', 'CI', 'HR', 'CU', 'CW', 'CY', 'CZ', 'DK', 'DJ', 'DM', 'DO', 'EC', 'EG', 'SV', 'GQ', 'ER', 'EE', 'ET', 'FK', 'FO', 'FJ', 'FI', 'FR', 'GF', 'PF', 'TF', 'GA', 'GM', 'GE', 'DE', 'GH', 'GI', 'GR', 'GL', 'GD', 'GP', 'GU', 'GT', 'GG', 'GN', 'GW', 'GY', 'HT', 'HM', 'VA', 'HN', 'HU', 'IS', 'IN', 'ID', 'IR', 'IQ', 'IE', 'IM', 'IL', 'IT', 'JM', 'JP', 'JE', 'JO', 'KZ', 'KE', 'KI', 'KP', 'KR', 'KO', 'KW', 'KG', 'LA', 'LV', 'LB', 'LS', 'LR', 'LY', 'LI', 'LT', 'LU', 'MK', 'MG', 'MW', 'MY', 'MV', 'ML', 'MT', 'MH', 'MQ', 'MR', 'MU', 'YT', 'MX', 'FM', 'MD', 'MC', 'MN', 'ME', 'MS', 'MA', 'MZ', 'MM', 'NA', 'NR', 'NP', 'NL', 'AN', 'NC', 'NZ', 'NI', 'NE', 'NG', 'NU', 'NF', 'MP', 'NO', 'OM', 'PK', 'PW', 'PS', 'PA', 'PG', 'PY', 'PE', 'PH', 'PN', 'PL', 'PT', 'PR', 'QA', 'RE', 'RO', 'RU', 'RW', 'BL', 'SH', 'KN', 'LC', 'MF', 'PM', 'VC', 'WS', 'SM', 'ST', 'SA', 'SN', 'RS', 'SC', 'SL', 'SG', 'SX', 'SK', 'SI', 'SB', 'SO', 'ZA', 'GS', 'SS', 'ES', 'LK', 'ZZ', 'SD', 'SR', 'SJ', 'SZ', 'SE', 'CH', 'SY', 'TW', 'TJ', 'TZ', 'TH', 'TL', 'TG', 'TK', 'TO', 'TT', 'TN', 'TR', 'TM', 'TC', 'TV', 'UG', 'UA', 'AE', 'GB', 'US', 'UM', 'UY', 'UZ', 'VU', 'VE', 'VN', 'VI', 'WF', 'EH', 'YE', 'ZM', 'ZW'))
            )"""
        )

checks["valid_appellantNationalitiesDescription_not_null"] = (
            """(
                (appellantNationalitiesDescription IS NOT NULL)
                AND
                (appellantNationalitiesDescription IN ('Afghanistan', 'Aland Islands', 'Albania', 'Algeria', 'American Samoa', 'Andorra', 'Angola', 'Anguilla', 'Antarctica', 'Antigua and Barbuda', 'Argentina', 'Armenia', 'Aruba', 'Australia', 'Austria', 'Azerbaijan', 'Bahamas', 'Bahrain', 'Bangladesh', 'Barbados', 'Belarus', 'Belgium', 'Belize', 'Benin', 'Bermuda', 'Bhutan', 'Bolivia', 'Bonaire, Sint Eustatius and Saba', 'Bosnia and Herzegovina', 'Botswana', 'Bouvet Island', 'Brazil', 'British Overseas Citizen', 'British Virgin Islands', 'British Indian Ocean Territory', 'Brunei Darussalam', 'Bulgaria', 'Burkina Faso', 'Burundi', 'Cambodia', 'Cameroon', 'Canada', 'Cape Verde', 'Cayman Islands', 'Central African Republic', 'Chad', 'Chile', 'China', 'Hong Kong, Special Administrative Region of China', 'Macao, Special Administrative Region of China', 'Christmas Island', 'Cocos (Keeling) Islands', 'Colombia', 'Comoros', 'Congo (Brazzaville)', 'Congo, Democratic Republic of the', 'Cook Islands', 'Costa Rica', 'Côte d\\'Ivoire', 'Croatia', 'Cuba', 'Curaçao', 'Cyprus', 'Czech Republic', 'Denmark', 'Djibouti', 'Dominica', 'Dominican Republic', 'Ecuador', 'Egypt', 'El Salvador', 'Equatorial Guinea', 'Eritrea', 'Estonia', 'Ethiopia', 'Falkland Islands (Malvinas)', 'Faroe Islands', 'Fiji', 'Finland', 'France', 'French Guiana', 'French Polynesia', 'French Southern Territories', 'Gabon', 'Gambia', 'Georgia', 'Germany', 'Ghana', 'Gibraltar', 'Greece', 'Greenland', 'Grenada', 'Guadeloupe', 'Guam', 'Guatemala', 'Guernsey', 'Guinea', 'Guinea-Bissau', 'Guyana', 'Haiti', 'Heard Island and Mcdonald Islands', 'Holy See (Vatican City State)', 'Honduras', 'Hungary', 'Iceland', 'India', 'Indonesia', 'Iran, Islamic Republic of', 'Iraq', 'Ireland', 'Isle of Man', 'Israel', 'Italy', 'Jamaica', 'Japan', 'Jersey', 'Jordan', 'Kazakhstan', 'Kenya', 'Kiribati', 'Korea, Democratic People\\'s Republic of', 'Korea, Republic of', 'Kosovo', 'Kuwait', 'Kyrgyzstan', 'Lao PDR', 'Latvia', 'Lebanon', 'Lesotho', 'Liberia', 'Libya', 'Liechtenstein', 'Lithuania', 'Luxembourg', 'Macedonia, Republic of', 'Madagascar', 'Malawi', 'Malaysia', 'Maldives', 'Mali', 'Malta', 'Marshall Islands', 'Martinique', 'Mauritania', 'Mauritius', 'Mayotte', 'Mexico', 'Micronesia, Federated States of', 'Moldova', 'Monaco', 'Mongolia', 'Montenegro', 'Montserrat', 'Morocco', 'Mozambique', 'Myanmar', 'Namibia', 'Nauru', 'Nepal', 'Netherlands', 'Netherlands Antilles', 'New Caledonia', 'New Zealand', 'Nicaragua', 'Niger', 'Nigeria', 'Niue', 'Norfolk Island', 'Northern Mariana Islands', 'Norway', 'Oman', 'Pakistan', 'Palau', 'Palestinian Territory, Occupied', 'Panama', 'Papua New Guinea', 'Paraguay', 'Peru', 'Philippines', 'Pitcairn', 'Poland', 'Portugal', 'Puerto Rico', 'Qatar', 'Réunion', 'Romania', 'Russian Federation', 'Rwanda', 'Saint-Barthélemy', 'Saint Helena', 'Saint Kitts and Nevis', 'Saint Lucia', 'Saint-Martin (French part)', 'Saint Pierre and Miquelon', 'Saint Vincent and Grenadines', 'Samoa', 'San Marino', 'Sao Tome and Principe', 'Saudi Arabia', 'Senegal', 'Serbia', 'Seychelles', 'Sierra Leone', 'Singapore', 'Sint Maarten (Dutch part)', 'Slovakia', 'Slovenia', 'Solomon Islands', 'Somalia', 'South Africa', 'South Georgia and the South Sandwich Islands', 'South Sudan', 'Spain', 'Sri Lanka', 'Stateless', 'Sudan', 'Suriname *', 'Svalbard and Jan Mayen Islands', 'Swaziland', 'Sweden', 'Switzerland', 'Syrian Arab Republic (Syria)', 'Taiwan', 'Tajikistan', 'Tanzania *, United Republic of', 'Thailand', 'Timor-Leste', 'Togo', 'Tokelau', 'Tonga', 'Trinidad and Tobago', 'Tunisia', 'Turkey', 'Turkmenistan', 'Turks and Caicos Islands', 'Tuvalu', 'Uganda', 'Ukraine', 'United Arab Emirates', 'United Kingdom', 'United States of America', 'United States Minor Outlying Islands', 'Uruguay', 'Uzbekistan', 'Vanuatu', 'Venezuela (Bolivarian Republic of)', 'Viet Nam', 'Virgin Islands, US', 'Wallis and Futuna Islands', 'Western Sahara', 'Yemen', 'Zambia', 'Zimbabwe'))
            )"""
        )

#####################
#Get Cases Failing Validation
#####################
dq_rules = "({0})".format(" AND ".join(checks.values()))


for state_table in states_to_update:
    print(f"Processing State : {state_table}")
    df_validation_filtered = spark.sql(f"""
    SELECT *
    FROM {state_table}
    WHERE CaseNo NOT IN (
    SELECT
        CaseNo
    FROM {state_table}
    WHERE {dq_rules})
    """)
    df_validation_filtered = df_validation_filtered.select("CaseNo", "appellantNationalitiesDescription")


    #####################
    #join with bronze data
    #####################
    #Join Failed Record in Gold With Bronze
    df_validation_filtered = df_validation_filtered.join(M1_bronze, on="CaseNo", how="inner")
    display(df_validation_filtered.select("CaseNo",  "appellantNationalitiesDescription", "NationalityId"))

    #####################
    #Update Data
    #####################
    #Pull out case Numbers to patch
    case_nos_to_update = [row["CaseNo"] for row in df_validation_filtered.select("CaseNo").distinct().collect()]
    # --- patch diagnostic (auto) ---
    _pl = globals().setdefault("PATCH_LOG", [])
    _pl.append({"rule": ", ".join(checks.keys()), "table": bronze_table_to_update, "state": state_table, "cases_patched": len(case_nos_to_update)})
    print(f"   -> patched {len(case_nos_to_update)} case(s) for {', '.join(checks.keys())} in {state_table}")
    #set table to update
    bronze_table_to_update_df = DeltaTable.forName(spark, bronze_table_to_update)

    #Display Before
    display(M1_bronze.filter(col("CaseNo").isin(case_nos_to_update)).select("CaseNo", "NationalityId"))

    bronze_table_to_update_df.update(
        condition=col("CaseNo").isin(case_nos_to_update),
        set={
            "NationalityId" : lit(41)
             }
    )

    #Display After
    display(M1_bronze.filter(col("CaseNo").isin(case_nos_to_update)).select("CaseNo", "NationalityId"))

In [0]:
##############################
#valid_appellantDateOfBirth_format - patch : "CaseRep_Address1"
##############################


#Setup
checks = {}
bronze_table_to_update = "ariadm_active_appeals.bronze_appealcase_caseappellant_appellant"

#Add the Gold Tables to the list below for any States that need updating
state_to_update_2 = "appealsubmitted_gold.stg_main_appeal_submitted_validation"
state_to_update_3 = "awaitingrespondentevidencea_gold.stg_main_awaiting_respondent_evidence_a_validation"
state_to_update_4 = "awaitingrespondentevidenceb_gold.stg_main_awaiting_respondent_evidence_b_validation"
state_to_update_5 = "caseunderreview_gold.stg_main_case_under_review_validation"
state_to_update_6 = "reasonsForAppealSubmitted_gold.stg_main_reasons_for_appeal_submitted_validation"
state_to_update_7 = "listing_gold.stg_main_listing_validation"
state_to_update_9 = "remitted_gold.stg_main_remitted_validation"
states_to_update = [state_to_update_2, state_to_update_6,state_to_update_9]
# states_to_update = [state_to_update_2, state_to_update_3,state_to_update_4]

checks["valid_appellantDateOfBirth_format"] = (
        "(appellantDateOfBirth IS NOT NULL AND appellantDateOfBirth RLIKE r'^\\d{4}-\\d{2}-\\d{2}$')"
    )

#####################
#Get Cases Failing Validation
#####################
dq_rules = "({0})".format(" AND ".join(checks.values()))


for state_table in states_to_update:
    print(f"Processing State : {state_table}")
    df_validation_filtered = spark.sql(f"""
    SELECT *
    FROM {state_table}
    WHERE CaseNo NOT IN (
    SELECT
        CaseNo
    FROM {state_table}
    WHERE {dq_rules})
    """)
    df_validation_filtered = df_validation_filtered.select("CaseNo", "appellantDateOfBirth")


    #####################
    #join with bronze data
    #####################
    #Join Failed Record in Gold With Bronze
    df_validation_filtered = df_validation_filtered.join(M2_bronze, on="CaseNo", how="inner")
    display(df_validation_filtered.select("CaseNo",  "appellantDateOfBirth", "BirthDate"))

    #####################
    #Update Data
    #####################
    #Pull out case Numbers to patch
    case_nos_to_update = [row["CaseNo"] for row in df_validation_filtered.select("CaseNo").distinct().collect()]
    # --- patch diagnostic (auto) ---
    _pl = globals().setdefault("PATCH_LOG", [])
    _pl.append({"rule": ", ".join(checks.keys()), "table": bronze_table_to_update, "state": state_table, "cases_patched": len(case_nos_to_update)})
    print(f"   -> patched {len(case_nos_to_update)} case(s) for {', '.join(checks.keys())} in {state_table}")
    #set table to update
    bronze_table_to_update_df = DeltaTable.forName(spark, bronze_table_to_update)

    #Display Before    
    display(M2_bronze.filter(col("CaseNo").isin(case_nos_to_update)).select("CaseNo", "BirthDate"))    

    bronze_table_to_update_df.update(
        condition=col("CaseNo").isin(case_nos_to_update),
        set={
            "BirthDate" : lit("1963-03-18T00:00:00.000+00:00")                 
             }
    )

    #Display After
    display(M2_bronze.filter(col("CaseNo").isin(case_nos_to_update)).select("CaseNo", "BirthDate"))    

In [0]:
##############################
#valid_appellantInterpreterLanguageCategory   - patch : "LanguageId"
##############################


#Setup
checks = {}
bronze_table_to_update = "ariadm_active_appeals.bronze_appealcase_crep_rep_floc_cspon_cfs"

#Add the Gold Tables to the list below for any States that need updating
state_to_update_7 = "listing_gold.stg_main_listing_validation"
state_to_update_8 = "prepareforhearing_gold.stg_main_prepare_for_hearing_validation"
state_to_update_9 = "remitted_gold.stg_main_remitted_validation"
state_to_update_10 = "decideda_gold.stg_main_decided_a_validation"
state_to_update_13 = "ftpadecided_gold.stg_main_ftpadecided_validation"
#only seen on listing so far
states_to_update = [state_to_update_7,state_to_update_8,state_to_update_9, state_to_update_10, state_to_update_13]


checks["valid_appellantInterpreterLanguageCategory"] = (
        """(
            CASE
                WHEN (
                    (NOT(Interpreter <=> 1))
                    OR
                    (
                        (LanguageId IS NULL OR LanguageId <=> 0)
                        AND
                        (AdditionalLanguageId IS NULL OR AdditionalLanguageId <=> 0)
                    )
                ) THEN (
                    (appellantInterpreterLanguageCategory IS NULL)
                ) ELSE (
                    (
                        (LanguageId IS NULL OR LanguageId <=> 0)
                        OR
                        (LanguageId IS NOT NULL AND NOT(LanguageId <=> 0) AND ARRAY_CONTAINS(appellantInterpreterLanguageCategory, valid_languageCategory))
                    )
                    AND
                    (
                        (AdditionalLanguageId IS NULL OR AdditionalLanguageId <=> 0)
                        OR
                        (AdditionalLanguageId IS NOT NULL AND NOT(AdditionalLanguageId <=> 0) AND ARRAY_CONTAINS(appellantInterpreterLanguageCategory, valid_additionalLanguageCategory))
                    )
                )
            END
        )"""
)


#####################
#Get Cases Failing Validation
#####################
dq_rules = "({0})".format(" AND ".join(checks.values()))


for state_table in states_to_update:
    print(f"Processing State : {state_table}")
    df_validation_filtered = spark.sql(f"""
    SELECT *
    FROM {state_table}
    WHERE CaseNo NOT IN (
    SELECT
        CaseNo
    FROM {state_table}
    WHERE {dq_rules})
    """)
    df_validation_filtered = df_validation_filtered.select("CaseNo", "appellantInterpreterLanguageCategory")


    #####################
    #join with bronze data
    #####################
    #Join Failed Record in Gold With Bronze
    df_validation_filtered = df_validation_filtered.join(M2_bronze, on="CaseNo", how="inner")
    display(df_validation_filtered.select("CaseNo",  "appellantInterpreterLanguageCategory"))

    #####################
    #Update Data
    #####################
    #Pull out case Numbers to patch
    case_nos_to_update = [row["CaseNo"] for row in df_validation_filtered.select("CaseNo").distinct().collect()]
    # --- patch diagnostic (auto) ---
    _pl = globals().setdefault("PATCH_LOG", [])
    _pl.append({"rule": ", ".join(checks.keys()), "table": bronze_table_to_update, "state": state_table, "cases_patched": len(case_nos_to_update)})
    print(f"   -> patched {len(case_nos_to_update)} case(s) for {', '.join(checks.keys())} in {state_table}")
    #set table to update
    bronze_table_to_update_df = DeltaTable.forName(spark, bronze_table_to_update)

    #Display Before    
    display(M1_bronze.filter(col("CaseNo").isin(case_nos_to_update)).select("CaseNo", "LanguageId"))    

    bronze_table_to_update_df.update(
    condition=col("CaseNo").isin(case_nos_to_update),
        set={
            "LanguageId" : lit("1")                 
             }
    )

    #Display After
    display(M1_bronze.filter(col("CaseNo").isin(case_nos_to_update)).select("CaseNo", "LanguageId"))    


In [0]:
##############################
#valid_sponsorGivenNames_not_null - patch : "Sponsor_Forenames"  (DIRECT bronze patch, no gold read)
# sponsorGivenNames is populated from Sponsor_Forenames when a sponsor exists. Obfuscation left
# Sponsor_Name populated but Sponsor_Forenames null -> sponsorGivenNames null -> rule FALSE.
# Patch a forename ONLY where Sponsor_Name is present and Sponsor_Forenames is null (never touches the
# Sponsor_Name-null side, so the rule's bidirectional null branch stays satisfied). Direct bronze filter
# avoids the stale-gold miss the gold-driven find suffered from.
##############################

bronze_table_to_update = "ariadm_active_appeals.bronze_appealcase_crep_rep_floc_cspon_cfs"
sponsor_bt = DeltaTable.forName(spark, bronze_table_to_update)
sponsor_bs = spark.table(bronze_table_to_update)

patch_cond = col("Sponsor_Name").isNotNull() & col("Sponsor_Forenames").isNull()

#Display Before
display(sponsor_bs.filter(patch_cond).select("CaseNo", "Sponsor_Name", "Sponsor_Forenames"))

_n = sponsor_bs.filter(patch_cond).select("CaseNo").distinct().count()

sponsor_bt.update(
    condition=patch_cond,
    set={
        "Sponsor_Forenames" : lit("JohnX")
         }
)

#Display After
display(sponsor_bs.filter(col("Sponsor_Name").isNotNull()).select("CaseNo", "Sponsor_Name", "Sponsor_Forenames"))

_pl = globals().setdefault("PATCH_LOG", [])
_pl.append({"rule": "valid_sponsorGivenNames_not_null", "table": bronze_table_to_update, "state": "ALL (direct bronze)", "cases_patched": _n})
print(f"   -> patched Sponsor_Forenames for {_n} case(s) (Sponsor_Name present, Sponsor_Forenames null)")


In [0]:
##############################
#valid_ftpaAppellantApplicationDate - patch : bronze_status DateReceived  (M3 = bronze_status_htype_clist_list_ltype_court_lsitting_adj)
# Spec (ARIA->CCD): ftpaAppellantApplicationDate = M3.DateReceived at MAX(StatusID) WHERE CaseStatus=39,
#   IF Party=1 -> include; ELSE Party=2 -> OMIT. ISO 8601. So we patch DateReceived on the CaseStatus=39
#   row for the affected case(s). Targeted (1 case this cut: IA/03566/2021, state remitted).
# NOTE: placeholder date below - BA to confirm the actual value ("Column I" / feedback).
##############################

cases_to_patch = ["IA/03566/2021"]
ftpa_bronze_status = "ariadm_active_appeals.bronze_status_htype_clist_list_ltype_court_lsitting_adj"
ftpa_date_value = "2020-01-01T00:00:00.000+00:00"   # <-- BA to set the real DateReceived

bt = DeltaTable.forName(spark, ftpa_bronze_status)
bs = spark.table(ftpa_bronze_status)

#Display Before (the CaseStatus=39 / Party=1 status rows for these cases)
display(bs.filter(col("CaseNo").isin(cases_to_patch) & (col("CaseStatus") == 39)).select("CaseNo", "CaseStatus", "Party", "DateReceived"))

bt.update(
    condition=(col("CaseNo").isin(cases_to_patch)) & (col("CaseStatus") == 39) & (col("Party") == 1),
    set={
        "DateReceived" : lit(ftpa_date_value)
         }
)

#Display After
display(bs.filter(col("CaseNo").isin(cases_to_patch) & (col("CaseStatus") == 39)).select("CaseNo", "CaseStatus", "Party", "DateReceived"))

#Log to the patch report
_n = bs.filter(col("CaseNo").isin(cases_to_patch) & (col("CaseStatus") == 39) & (col("Party") == 1)).select("CaseNo").distinct().count()
_pl = globals().setdefault("PATCH_LOG", [])
_pl.append({"rule": "valid_ftpaAppellantApplicationDate", "table": ftpa_bronze_status, "state": "remitted", "cases_patched": _n})
print(f"   -> patched DateReceived (CaseStatus=39, Party=1) for {_n} case(s): {cases_to_patch}")

In [0]:
cases = ["EA/13092/2021","EA/13083/2021","EA/13090/2021","EA/13822/2021","EA/01617/2023","HU/01202/2022","EA/13078/2021","EA/06952/2022",
        "EA/06966/2021","DA/00243/2020","HU/00575/2025","EA/07366/2021","EA/11255/2021","EA/00554/2025"]

cases = ["LP/00921/2023"]

# df = M2_bronze.filter(col("CaseNo").isin(cases))
# display(df.select("CaseNo","AppellantCountryId" ,"Appellant_Address1", "Appellant_Address2", "Appellant_Address3", "Appellant_Address4", "Appellant_Address5", "Appellant_Postcode"))

# display(M1_bronze.select("CaseNo",  "countryGovUkOocAdminJ", "Appellant_Address1", "Appellant_Address2", "Appellant_Address3", "Appellant_Address4", "Appellant_Address5", "Appellant_Postcode", ))

# display(M2_bronze.select("CaseNo",  "Appellant_Address1", "Appellant_Address2", "Appellant_Address3", "Appellant_Address4", "Appellant_Address5", "Appellant_Postcode", ))
# display(M2_bronze.select("CaseNo", "BirthDate"))

display(M1_bronze)

In [0]:
# =============================================================================
# PATCH REPORT - collected from every patch cell above.
# Tells you, per rule/field, whether bronze data was actually patched this run,
# prints it, and writes an xlsx to the Results folder (path printed at the end).
# =============================================================================
import pandas as pd, os, datetime

PATCH_LOG = globals().get("PATCH_LOG", [])

# dedupe: keep the last record per (rule, table, state) in case a cell was re-run
_seen = {}
for e in PATCH_LOG:
    _seen[(e["rule"], e["table"], e["state"])] = e
rows = list(_seen.values())

cols = ["rule", "table", "state", "cases_patched"]
detail = pd.DataFrame(rows, columns=cols) if rows else pd.DataFrame(columns=cols)

if len(detail):
    summary = (detail.groupby(["rule", "table"], as_index=False)
                     .agg(states_processed=("state", "nunique"),
                          states_with_patches=("cases_patched", lambda s: int((s > 0).sum())),
                          total_cases_patched=("cases_patched", "sum")))
    summary["status"] = summary["total_cases_patched"].apply(
        lambda n: f"PATCHED ({int(n)})" if n > 0 else "NOTHING TO PATCH")
    summary = summary.sort_values("total_cases_patched", ascending=False)
else:
    summary = pd.DataFrame(columns=["rule", "table", "states_processed",
                                    "states_with_patches", "total_cases_patched", "status"])

print("=" * 78)
print("BRONZE DATA UPDATER - PATCH REPORT")
print("=" * 78)
if len(summary):
    print(summary.to_string(index=False))
    print("\nTOTAL cases patched this run:", int(summary["total_cases_patched"].sum()))
    _none = summary[summary["total_cases_patched"] == 0]["rule"].tolist()
    if _none:
        print("NOTHING patched for:", _none)
else:
    print("PATCH_LOG is empty - run all patch cells (Run All) first.")

# write to Results
try:
    _user = spark.sql("select current_user()").first()[0]
    OUT = f"/Workspace/Users/{_user}/Results/bronze_data_updater/" + datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    os.makedirs(OUT, exist_ok=True)
    try:
        import openpyxl  # noqa
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl"])
    xlsx = OUT + "/patch_report.xlsx"
    with pd.ExcelWriter(xlsx, engine="openpyxl") as w:
        summary.to_excel(w, sheet_name="Summary", index=False)
        detail.to_excel(w, sheet_name="PerState", index=False)
    print("\nReport written:", xlsx)
    print("Folder        :", OUT)
except Exception as _e:
    print("\nreport write failed:", _e)